# TP4 — Fêtes et consommation au Sénégal (2016–2025)
**Python avancé · L3 IA · ISM**  
Binôme : Étudiant 1 ____________________ · Étudiant 2 ____________________  
Enseignant : Elias BADJI

> Données fictives à usage pédagogique. Les conclusions portent uniquement sur ces données simulées et ne doivent pas être généralisées à une communauté réelle.

## Méthode générale
Le notebook suit l'ordre de l'énoncé. Les ventes de 2016–2023 servent à l'entraînement, 2024 à la validation et 2025 au test final. Pour les modèles de prévision, aucune vente récente n'est utilisée comme variable d'entrée. Les fichiers CSV sont recherchés dans le même dossier que ce notebook.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             silhouette_score, mean_absolute_error)
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from IPython.display import display

RANDOM_STATE = 42
BASE = Path.cwd()
# Le notebook fonctionne aussi si les CSV sont dans /mnt/data.
def locate(filename):
    candidates = [BASE / filename, Path("/mnt/data") / filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Fichier introuvable : {filename}. Placez-le à côté du notebook.")

sales = pd.read_csv(locate("ventes_journalieres(1).csv"), parse_dates=["date"])
households = pd.read_csv(locate("enquete_menages(1).csv"))
calendar = pd.read_csv(locate("calendrier_evenements.csv"), parse_dates=["date_debut", "date_fin"])
shops = pd.read_csv(locate("magasins.csv"), parse_dates=["date_ouverture"])
categories = pd.read_csv(locate("categories.csv"))
print("Ventes :", sales.shape, "| Enquête :", households.shape, "| Calendrier :", calendar.shape)

## Partie 1 — Audit et nettoyage (3 points)
Audit chiffré des doublons, catégories, valeurs manquantes, quantités négatives/aberrantes et couverture calendaire. Les pics proches des fêtes ne sont pas supprimés automatiquement : une règle purement statistique pourrait effacer le signal recherché.

In [ ]:
# Audit brut
sales["categorie_brute"] = sales["categorie"]
sales["categorie_norm"] = sales["categorie"].astype(str).str.strip().str.lower()
valid_categories = set(categories["categorie"].str.strip().str.lower())
audit = {
    "lignes": len(sales),
    "doublons_exactement": int(sales.duplicated().sum()),
    "doublons_cle_date_magasin_categorie": int(sales.assign(categorie_norm=sales["categorie_norm"]).duplicated(["date","magasin_id","categorie_norm"]).sum()),
    "categories_brutes_distinctes": int(sales["categorie"].nunique()),
    "lignes_categorie_mal_saisie": int((~sales["categorie_norm"].isin(valid_categories)).sum()),
    "quantite_manquante": int(sales["quantite"].isna().sum()),
    "ca_manquant": int(sales["chiffre_affaires_fcfa"].isna().sum()),
    "quantite_negative": int((sales["quantite"] < 0).sum()),
    "prix_non_positif": int((sales["prix_unitaire_fcfa"] <= 0).sum()),
}
audit_df = pd.Series(audit, name="Valeur").to_frame()
display(audit_df)
# Valeurs aberrantes par catégorie : règle IQR, à examiner et non à supprimer automatiquement.
q = sales.groupby("categorie_norm")["quantite"].quantile([.25,.75]).unstack()
q.columns = ["Q1","Q3"]
q["borne_sup_IQR"] = q["Q3"] + 3*(q["Q3"]-q["Q1"])
sales = sales.merge(q[["borne_sup_IQR"]], left_on="categorie_norm", right_index=True, how="left")
sales["aberrante_IQR"] = sales["quantite"] > sales["borne_sup_IQR"]
print("Quantités positives au-dessus de Q3 + 3×IQR :", int(sales["aberrante_IQR"].sum()))
# Jours manquants : comparer à la grille calendrier de chaque magasin depuis sa date d'ouverture.
daily_counts = sales.groupby("magasin_id")["date"].agg(["min","max","nunique"])
coverage = []
for r in shops.itertuples():
    observed = set(sales.loc[sales.magasin_id.eq(r.magasin_id), "date"].dt.normalize())
    expected = pd.date_range(max(pd.Timestamp("2016-01-01"), r.date_ouverture), "2025-12-31", freq="D")
    missing_days = sum(d not in observed for d in expected)
    coverage.append({"magasin_id":r.magasin_id,"ville":r.ville,"jours_attendus":len(expected),
                     "jours_observes":len(observed),"jours_absents":missing_days})
coverage = pd.DataFrame(coverage)
display(coverage)
# Nettoyage : normalisation des catégories, suppression doublons de clé, imputation prudente.
sales["categorie"] = sales["categorie_norm"]
sales = sales[sales["categorie"].isin(valid_categories)].copy()
sales = sales.sort_values(["date","magasin_id","categorie"]).drop_duplicates(
    ["date","magasin_id","categorie"], keep="first")
# Une quantité/CA manquants sont imputés par la médiane catégorie×magasin, puis catégorie.
for col in ["quantite","chiffre_affaires_fcfa"]:
    sales[col] = sales[col].fillna(sales.groupby(["magasin_id","categorie"])[col].transform("median"))
    sales[col] = sales[col].fillna(sales.groupby("categorie")[col].transform("median"))
sales["prix_unitaire_fcfa"] = sales["prix_unitaire_fcfa"].where(sales["prix_unitaire_fcfa"] > 0)
sales["prix_unitaire_fcfa"] = sales["prix_unitaire_fcfa"].fillna(
    sales.groupby("categorie")["prix_unitaire_fcfa"].transform("median"))
sales["promo"] = sales["promo"].fillna(0).astype(int)
sales["part_paiement_mobile"] = sales["part_paiement_mobile"].clip(0,1)
sales = sales.drop(columns=["categorie_brute","categorie_norm","borne_sup_IQR","aberrante_IQR"], errors="ignore")
print("Après nettoyage :", sales.shape, "| valeurs manquantes quantité :", sales.quantite.isna().sum())

**Justification des traitements.** Les catégories sont harmonisées en minuscules et sans espaces pour réunir les variantes de casse. Les doublons sur la clé date–magasin–catégorie sont ramenés à une seule ligne afin d'éviter de compter deux fois la même vente. Les quantités et chiffres d'affaires manquants sont imputés par une médiane locale, sans remplacer les pics festifs. Les valeurs négatives, si présentes, doivent être vérifiées comme possibles retours/erreurs avant suppression ; elles sont signalées dans l'audit. Les jours absents sont rapportés par magasin : l'absence d'une ligne n'est pas automatiquement considérée comme zéro vente sans confirmation du fonctionnement du point de vente.

## Partie 2 — Variables d'événement (3 points)
La fonction ci-dessous construit pour chaque date la distance à la prochaine et à la précédente occurrence de chaque événement, en utilisant `merge_asof` plutôt qu'une boucle sur les ventes. Les distances sont plafonnées à 365 jours. Ramadan est représenté par un indicateur de présence entre début et fin.

In [ ]:
def variables_calendrier(dates, calendrier=calendar, plafond=365):
    # Calculer une seule fois par date distincte, puis rétablir l'ordre d'origine.
    dates_originales = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    uniques = pd.DataFrame({"date": dates_originales.dropna().drop_duplicates().sort_values()})
    out = uniques.copy()
    slug = {"é":"e","è":"e","ê":"e","û":"u","ô":"o","à":"a","ï":"i"}
    for ev in calendrier["evenement"].dropna().unique():
        evcal = calendrier.loc[calendrier["evenement"].eq(ev)].sort_values("date_debut")
        occ = pd.DataFrame({"date_evenement": pd.to_datetime(evcal["date_debut"].dropna().drop_duplicates())}).sort_values("date_evenement")
        prev = pd.merge_asof(uniques, occ, left_on="date", right_on="date_evenement", direction="backward")
        av_prev = (prev["date"] - prev["date_evenement"]).dt.days.clip(upper=plafond)
        nxt = pd.merge_asof(uniques, occ, left_on="date", right_on="date_evenement", direction="forward")
        av_next = (nxt["date_evenement"] - nxt["date"]).dt.days.clip(upper=plafond)
        name = "".join(slug.get(ch,ch) for ch in ev.lower())
        name = "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")
        out[f"ap_{name}"] = av_prev.to_numpy()
        out[f"av_{name}"] = av_next.to_numpy()
    ram = calendrier[calendrier.evenement.eq("Ramadan")][["date_debut","date_fin"]].dropna().copy()
    ram["date_debut"] = pd.to_datetime(ram.date_debut); ram["date_fin"] = pd.to_datetime(ram.date_fin)
    # Intervalle Ramadan vectorisé, sans apply sur les centaines de milliers de ventes.
    ramadan = np.zeros(len(uniques), dtype=int)
    for rr in ram.itertuples(index=False):
        ramadan |= ((uniques.date >= rr.date_debut) & (uniques.date <= rr.date_fin)).to_numpy()
    out["ramadan"] = ramadan.astype(int)
    result = dates_originales.to_frame(name="date").merge(out, on="date", how="left", sort=False)
    return result


**Conclusion partie 2.** Une variable « mois » ne suffit pas : les fêtes musulmanes suivent un calendrier lunaire et se déplacent dans l'année grégorienne. Deux dates du même mois peuvent être à des distances très différentes de la Tabaski ou de la Korité. Les distances `av_…` et `ap_…` rendent ce signal explicite.

## Partie 3 — Mesurer l'effet des fêtes (4 points)
L'uplift est calculé comme moyenne journalière de J−7 à J−1 divisée par celle de J−60 à J−31. Pour le Magal et le Maouloud, les résultats sont détaillés par magasin et ville. La carte de chaleur montre les catégories avec le plus grand uplift national.

In [ ]:
# Agrégation catégorie-jour ; garder une clé date propre.
daily = sales.groupby(["date","magasin_id","categorie"], as_index=False).agg(
    quantite=("quantite","sum"), ca=("chiffre_affaires_fcfa","sum"),
    prix=("prix_unitaire_fcfa","median"), mobile=("part_paiement_mobile","mean"))
daily = daily.merge(shops[["magasin_id","ville","region"]], on="magasin_id", how="left")
event_rows = calendar[calendar.evenement.ne("Ramadan")].copy()
event_rows["date_debut"] = pd.to_datetime(event_rows["date_debut"])
uplift_rows=[]
for ev in event_rows.itertuples():
    w0, w1 = ev.date_debut-pd.Timedelta(days=7), ev.date_debut-pd.Timedelta(days=1)
    r0, r1 = ev.date_debut-pd.Timedelta(days=60), ev.date_debut-pd.Timedelta(days=31)
    d = daily[(daily.date>=r0)&(daily.date<=r1)]
    ref = d.groupby("categorie").quantite.mean()
    d2 = daily[(daily.date>=w0)&(daily.date<=w1)]
    win = d2.groupby("categorie").quantite.mean()
    for cat in sorted(set(ref.index)|set(win.index)):
        if cat in ref.index and cat in win.index and ref[cat] > 0:
            uplift_rows.append({"evenement":ev.evenement,"annee":ev.annee,"categorie":cat,
                                "moyenne_fenetre":win[cat],"moyenne_reference":ref[cat],
                                "uplift":win[cat]/ref[cat]})
uplifts=pd.DataFrame(uplift_rows)
top3 = (uplifts.groupby(["evenement","categorie"]).uplift.mean().reset_index()
        .sort_values(["evenement","uplift"],ascending=[True,False])
        .groupby("evenement").head(3))
display(top3.sort_values(["evenement","uplift"],ascending=[True,False]))
pivot = uplifts.groupby(["evenement","categorie"]).uplift.mean().unstack()
plt.figure(figsize=(14,7))
sns.heatmap(np.log2(pivot.clip(lower=.1)), cmap="YlOrRd", center=0)
plt.title("Uplift moyen par événement et catégorie (échelle couleur = log2 de l'uplift)")
plt.xlabel("Catégorie"); plt.ylabel("Événement"); plt.tight_layout(); plt.show()
# Magal vs Maouloud, par ville / point de vente
# Comparaison plus directe via les uplifts sur les fenêtres de chaque événement.
local_rows=[]
for ev in ["Magal de Touba","Maouloud"]:
    for e in event_rows[event_rows.evenement.eq(ev)].itertuples():
        for store in shops.itertuples():
            sub=daily[daily.magasin_id.eq(store.magasin_id)]
            win=sub[(sub.date>=e.date_debut-pd.Timedelta(days=7))&(sub.date<=e.date_debut-pd.Timedelta(days=1))].groupby("categorie").quantite.mean()
            ref=sub[(sub.date>=e.date_debut-pd.Timedelta(days=60))&(sub.date<=e.date_debut-pd.Timedelta(days=31))].groupby("categorie").quantite.mean()
            ratio=(win/ref.replace(0,np.nan)).replace([np.inf,-np.inf],np.nan).mean()
            if pd.notna(ratio): local_rows.append({"evenement":ev,"annee":e.annee,"magasin_id":store.magasin_id,"ville":store.ville,"uplift_moyen":ratio})
local=pd.DataFrame(local_rows)
display(local.groupby(["evenement","ville"]).uplift_moyen.mean().unstack("evenement").round(2))
# Indice des prix construit à partir du panier de catégories pondéré par CA 2016.
price = sales.assign(annee=sales.date.dt.year).groupby(["annee","categorie"]).prix_unitaire_fcfa.median().unstack()
ca16 = sales[sales.date.dt.year.eq(2016)].groupby("categorie").chiffre_affaires_fcfa.sum()
weights = ca16 / ca16.sum()
base_price = price.loc[2016]
index = (price.div(base_price, axis=1).mul(weights, axis=1)).sum(axis=1)
trend = sales.assign(annee=sales.date.dt.year).groupby("annee").agg(ca_nominal=("chiffre_affaires_fcfa","sum"),mobile=("part_paiement_mobile","mean"))
trend["indice_prix_2016_100"] = index*100
trend["ca_reel_base_2016"] = trend.ca_nominal/index
display(trend.round(2))
# Prix mouton autour de la Tabaski, et part moyenne mobile
mouton = sales[(sales.categorie.eq("betail_mouton"))].copy()
mouton["annee"]=mouton.date.dt.year
print("Prix médian du bétail/mouton par année :")
display(mouton.groupby("annee").prix_unitaire_fcfa.median().rename("prix_median_fcfa").to_frame())
trend[["ca_nominal","ca_reel_base_2016"]].plot(marker="o", figsize=(9,4), title="Chiffre d'affaires nominal et réel (base 2016)")
plt.tight_layout(); plt.show()

**Précautions d'interprétation.** L'uplift est descriptif, pas une preuve de causalité. Un événement peut se trouver dans la fenêtre de référence d'un autre événement ; par exemple, des événements de fin d'année peuvent se rapprocher de Noël ou de Maouloud selon l'année. Les fenêtres qui se chevauchent attribuent potentiellement le même surcroît à plusieurs événements. Il faut interpréter les ratios élevés avec le calendrier et les volumes de base. L'indice de prix ci-dessus est un indice pédagogique construit à partir des prix médians et des poids de CA 2016, comme demandé dans l'annexe.

## Partie 4 — Prévoir la demande un an à l'avance (5 points)
La cible est la quantité journalière par magasin et catégorie. Les entrées sont limitées aux variables calendaires, au magasin, à la catégorie et à la promotion planifiée. Les ventes des jours précédents sont exclues. WAPE = somme des erreurs absolues / somme des ventes réelles.

In [ ]:
# Créer une grille observée date-magasin-catégorie ; variables connues à l'avance uniquement.
model_df = daily[["date","magasin_id","categorie","quantite"]].copy()
model_df["annee"] = model_df.date.dt.year
model_df["mois"] = model_df.date.dt.month
model_df["jour_annee"] = model_df.date.dt.dayofyear
model_df["jour_semaine"] = model_df.date.dt.dayofweek
model_df["weekend"] = (model_df.jour_semaine>=5).astype(int)
model_df = model_df.merge(sales.groupby(["date","magasin_id","categorie"], as_index=False).promo.max(),
                          on=["date","magasin_id","categorie"], how="left")
# Calendrier event features; features distances are based only on dates/events known in advance.
event_feat = variables_calendrier(model_df["date"])
model_df = pd.concat([model_df.reset_index(drop=True), event_feat.drop(columns="date").reset_index(drop=True)], axis=1)
# Store/cat metadata
model_df = model_df.merge(shops[["magasin_id","ville","type_point_vente","milieu"]], on="magasin_id", how="left")
model_df = model_df.merge(categories[["categorie","famille","unite"]], on="categorie", how="left")
distance_cols=[c for c in model_df.columns if c.startswith(("av_","ap_"))]+["ramadan"]
numeric_features=["annee","mois","jour_annee","jour_semaine","weekend","promo"]+distance_cols
categorical_features=["magasin_id","categorie","ville","type_point_vente","milieu","famille","unite"]
features=numeric_features+categorical_features
def wape(y, pred):
    den=np.sum(np.abs(np.asarray(y)))
    return np.sum(np.abs(np.asarray(y)-np.asarray(pred)))/den if den else np.nan
train=model_df[model_df.annee.between(2016,2023)].copy()
valid=model_df[model_df.annee.eq(2024)].copy()
test=model_df[model_df.annee.eq(2025)].copy()
print("Taille train/validation/test :",len(train),len(valid),len(test))
# Deux références naïves : 364 jours auparavant ; moyenne 2024 par magasin/catégorie/jour semaine.
key=["magasin_id","categorie","jour_semaine"]
lookup364=model_df[["date","magasin_id","categorie","quantite"]].copy()
lookup364["date"]=lookup364.date+pd.Timedelta(days=364)
lookup364=lookup364.rename(columns={"quantite":"pred_364"})
test=test.merge(lookup364,on=["date","magasin_id","categorie"],how="left")
avg24=valid.groupby(key).quantite.mean().rename("pred_moyenne24").reset_index()
test=test.merge(avg24,on=key,how="left")
naive_results=pd.DataFrame({
    "méthode":["Même jour de semaine, 364 jours avant","Moyenne 2024 par magasin/catégorie/jour semaine"],
    "WAPE":[wape(test.quantite,test.pred_364.fillna(0)),wape(test.quantite,test.pred_moyenne24.fillna(0))]
})
display(naive_results)
# Pipeline Ridge : imputation + OneHot + standardisation numérique, cible log1p.
pre = ColumnTransformer([
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric_features),
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical_features)
])
ridge=Pipeline([("pre",pre),("model",Ridge(alpha=10.0))])
ridge.fit(train[features],np.log1p(train.quantite.clip(lower=0)))
pred_ridge=np.expm1(ridge.predict(valid[features])).clip(0)
# HGB uses encoded features, validation selects loss/features. Use one-hot encoded sparse -> dense numeric code for speed.
# For HGB, categorical variables are encoded as category codes learned from combined train/valid categories;
# no target information is used in encoding.
hgb_df=model_df.copy()
hgb_features=numeric_features+categorical_features
for c in categorical_features:
    hgb_df[c]=hgb_df[c].astype("category").cat.codes
hgb_train=hgb_df[hgb_df.annee.between(2016,2023)]
hgb_valid=hgb_df[hgb_df.annee.eq(2024)]
hgb_test=hgb_df[hgb_df.annee.eq(2025)]
hgb_num_features=hgb_features
hgb_candidates={}
for loss in ["squared_error","poisson"]:
    m=HistGradientBoostingRegressor(loss=loss,max_iter=45,max_leaf_nodes=20,
                                    learning_rate=0.08,l2_regularization=1.0,
                                    random_state=RANDOM_STATE)
    yfit=np.log1p(hgb_train.quantite.clip(lower=0)) if loss=="squared_error" else hgb_train.quantite.clip(lower=0)
    m.fit(hgb_train.sample(n=min(180000,len(hgb_train)), random_state=RANDOM_STATE)[hgb_num_features], yfit.loc[hgb_train.sample(n=min(180000,len(hgb_train)), random_state=RANDOM_STATE).index])
    pv=np.expm1(m.predict(hgb_valid[hgb_num_features])).clip(0) if loss=="squared_error" else m.predict(hgb_valid[hgb_num_features]).clip(0)
    hgb_candidates[loss]=(m,wape(hgb_valid.quantite,pv),pv)
print("WAPE validation Ridge log1p :",round(wape(valid.quantite,pred_ridge),4))
print("WAPE validation HGB :",{k:round(v[1],4) for k,v in hgb_candidates.items()})
best_loss=min(hgb_candidates,key=lambda k:hgb_candidates[k][1])
best_hgb=hgb_candidates[best_loss][0]
print("Perte HGB choisie sur validation :",best_loss)

**Choix méthodologique.** La régression Ridge utilise une cible `log1p` pour réduire l'influence des grands volumes et un `Pipeline` avec `ColumnTransformer`, imputation, mise à l'échelle et encodage. Pour le modèle d'arbres, les pertes `squared_error` sur log1p et `poisson` sont comparées sur la validation 2024. La perte Poisson est adaptée aux comptages non négatifs ; la sélection finale repose sur le WAPE de validation, pas sur le test.

In [ ]:
# Réentraînement final sur 2016-2024, évaluation unique sur 2025.
train_final=model_df[model_df.annee.between(2016,2024)].copy()
test_final=model_df[model_df.annee.eq(2025)].copy()
ridge_final=clone(ridge).fit(train_final[features],np.log1p(train_final.quantite.clip(lower=0)))
pr=np.expm1(ridge_final.predict(test_final[features])).clip(0)
# Encode using category codes from complete feature table (feature-only transform, no target use).
hf_train=hgb_df[hgb_df.annee.between(2016,2024)]
hf_test=hgb_df[hgb_df.annee.eq(2025)]
hgb_final=HistGradientBoostingRegressor(loss=best_loss,max_iter=45,max_leaf_nodes=20,
                                        learning_rate=.08,l2_regularization=1.0,random_state=RANDOM_STATE)
yhf=np.log1p(hf_train.quantite.clip(lower=0)) if best_loss=="squared_error" else hf_train.quantite.clip(lower=0)
hf_sample=hf_train.sample(n=min(200000,len(hf_train)),random_state=RANDOM_STATE)
yhf_sample=np.log1p(hf_sample.quantite.clip(lower=0)) if best_loss=="squared_error" else hf_sample.quantite.clip(lower=0)
hgb_final.fit(hf_sample[hgb_num_features],yhf_sample)
ph=np.expm1(hgb_final.predict(hf_test[hgb_num_features])).clip(0) if best_loss=="squared_error" else hgb_final.predict(hf_test[hgb_num_features]).clip(0)
scores=pd.DataFrame({"Modèle":["Ridge log1p","HGB "+best_loss],"WAPE 2025":[wape(test_final.quantite,pr),wape(test_final.quantite,ph)]})
display(scores)
# WAPE par événement, dans les 14 jours précédant chaque fête en 2025
def event_window_wape(pred, name):
    tmp=test_final[["date","quantite"]].copy(); tmp["pred"]=pred
    rows=[]
    evs=calendar[(calendar.annee.eq(2025))&(calendar.evenement.ne("Ramadan"))]
    for e in evs.itertuples():
        dt=pd.Timestamp(e.date_debut)
        sub=tmp[(tmp.date>=dt-pd.Timedelta(days=14))&(tmp.date<dt)]
        if len(sub): rows.append({"evenement":e.evenement,"modele":name,"WAPE":wape(sub.quantite,sub.pred)})
    return rows
by_event=pd.DataFrame(event_window_wape(pr,"Ridge")+event_window_wape(ph,"HGB"))
display(by_event.pivot(index="evenement",columns="modele",values="WAPE").round(3))
# Three peaks: total real vs predicted in the 7 days preceding Tabaski, Magal Touba, Christmas.
peak_rows=[]
ev2025=calendar[(calendar.annee.eq(2025))&calendar.evenement.isin(["Tabaski","Magal de Touba","Noël"])]
for e in ev2025.itertuples():
    dt=pd.Timestamp(e.date_debut)
    mask=(test_final.date>=dt-pd.Timedelta(days=7))&(test_final.date<dt)
    idx=np.where(mask)[0]
    peak_rows.append({"événement":e.evenement,"ventes_réelles":test_final.quantite.iloc[idx].sum(),
                      "prévision_Ridge":pr[idx].sum(),"prévision_HGB":ph[idx].sum()})
peaks=pd.DataFrame(peak_rows)
display(peaks.round(1))
print("WAPE plus faible = meilleure précision globale. Une sous-estimation des pics est visible si le total prévu est inférieur au réel.")

## Partie 5 — Qui s'endette après la fête ? (3 points)
La cible est `endette_apres_fete`. Entraînement 2016–2023 et test 2024–2025. La religion et la confrérie sont exclues des variables prédictives de départ et réservées à l'analyse d'équité ; elles ne sont jamais livrées au modèle.

In [ ]:
hh=households.copy()
hh["annee"]=pd.to_numeric(hh.annee,errors="coerce")
spend=hh.groupby("evenement").agg(depense_mediane=("depense_totale_fcfa","median"),
                                 taux_endettement=("endette_apres_fete","mean"),n=("enquete_id","count"))
display(spend.sort_values("taux_endettement",ascending=False).round(3))
display(hh.groupby("financement_principal").agg(depense_mediane=("depense_totale_fcfa","median"),
    taux_endettement=("endette_apres_fete","mean"),n=("enquete_id","count")).sort_values("taux_endettement",ascending=False).round(3))
# Prevent leakage: exclude identifiers and target, plus sensitive variables from model.
target="endette_apres_fete"
exclude={"enquete_id","menage_id",target,"religion","confrerie"}
candidate=[c for c in hh.columns if c not in exclude]
# Exclude any free-form event ID-like field only if not useful; keep event, household profile, expenditure/financing.
X=hh[candidate].copy()
y=hh[target].astype(int)
cat_hh=X.select_dtypes(include=["object"]).columns.tolist()
num_hh=[c for c in X.columns if c not in cat_hh]
pre_hh=ColumnTransformer([
 ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_hh),
 ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore"))]),cat_hh)
])
trainmask=hh.annee.between(2016,2023)
testmask=hh.annee.between(2024,2025)
Xtr,Xte=X.loc[trainmask],X.loc[testmask]; ytr,yte=y.loc[trainmask],y.loc[testmask]
logit=Pipeline([("pre",pre_hh),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE))])
logit.fit(Xtr,ytr)
p_log=logit.predict_proba(Xte)[:,1]
# HGB uses imputed + onehot encoded dense/array; fit same preprocessor separately.
pre_hgb=clone(pre_hh)
Xtr_enc=pre_hgb.fit_transform(Xtr)
Xte_enc=pre_hgb.transform(Xte)
if hasattr(Xtr_enc,"toarray"): Xtr_enc=Xtr_enc.toarray(); Xte_enc=Xte_enc.toarray()
hclf=HistGradientBoostingClassifier(max_iter=60,max_leaf_nodes=15,learning_rate=.08,
                                    l2_regularization=1.0,random_state=RANDOM_STATE)

_hidx = np.random.RandomState(RANDOM_STATE).choice(len(ytr), size=min(18000, len(ytr)), replace=False)
hclf.fit(Xtr_enc[_hidx], ytr.iloc[_hidx])
p_hgb=hclf.predict_proba(Xte_enc)[:,1]
def cls_metrics(ytrue,p):
    return {"ROC AUC":roc_auc_score(ytrue,p),"PR AUC":average_precision_score(ytrue,p),
            "Brier":brier_score_loss(ytrue,p)}
class_scores=pd.DataFrame([{"Modèle":"Régression logistique",**cls_metrics(yte,p_log)},
                           {"Modèle":"HistGradientBoostingClassifier",**cls_metrics(yte,p_hgb)}])
display(class_scores.round(4))
best_name="Régression logistique" if class_scores.loc[0,"ROC AUC"]>=class_scores.loc[1,"ROC AUC"] else "HistGradientBoostingClassifier"
print("Modèle retenu pour l'analyse d'équité (ROC AUC) :",best_name)

In [ ]:
# Importance par permutation sur un sous-échantillon test pour limiter le coût de calcul.
rng=np.random.RandomState(RANDOM_STATE)
n=min(600,len(Xte)); sel=rng.choice(len(Xte),size=n,replace=False)
if best_name=="Régression logistique":
    estimator=logit
    perm=permutation_importance(estimator,Xte.iloc[sel],yte.iloc[sel],scoring="roc_auc",
                                n_repeats=1,random_state=RANDOM_STATE,n_jobs=1)
    imp=pd.DataFrame({"variable":Xte.columns,"importance_moyenne":perm.importances_mean})
else:
    perm=permutation_importance(hclf,Xte_enc[sel],yte.iloc[sel],scoring="roc_auc",
                                n_repeats=1,random_state=RANDOM_STATE,n_jobs=1)
    # For HGB encoded features, report encoded-column ranks.
    names=pre_hgb.get_feature_names_out()
    imp=pd.DataFrame({"variable":names,"importance_moyenne":perm.importances_mean})
display(imp.sort_values("importance_moyenne",ascending=False).head(12))
# Equité : comparer performance avec / sans variables sensibles.
# Les variables sensibles n'ont pas été utilisées dans le modèle de base; cette comparaison quantifie l'intérêt de les exclure.
sensitive_cols=[c for c in ["religion","confrerie"] if c in hh.columns]
X_sensitive=hh[[c for c in hh.columns if c not in {"enquete_id","menage_id",target}]].copy()
def fit_eval_without_sensitive(drop_sensitive):
    cols=[c for c in X_sensitive.columns if c not in drop_sensitive]
    xx=X_sensitive[cols]
    cats=xx.select_dtypes(include=["object"]).columns.tolist()
    nums=[c for c in xx.columns if c not in cats]
    prep=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),nums),
                            ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore"))]),cats)])
    model=Pipeline([("pre",prep),("clf",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE))])
    model.fit(xx.loc[trainmask],y.loc[trainmask])
    pp=model.predict_proba(xx.loc[testmask])[:,1]
    return {"variables_sensibles_exclues":", ".join(drop_sensitive) or "aucune",
            "ROC AUC":roc_auc_score(y.loc[testmask],pp),"PR AUC":average_precision_score(y.loc[testmask],pp),
            "Brier":brier_score_loss(y.loc[testmask],pp)}
equity=pd.DataFrame([fit_eval_without_sensitive([]),fit_eval_without_sensitive(sensitive_cols)])
display(equity.round(4))

**Conclusion équité (5–10 lignes).** Les métriques globales ne suffisent pas à juger si un modèle est juste : il faut aussi examiner les erreurs et la calibration entre groupes, avec prudence et des effectifs suffisants. La religion et la confrérie sont des variables sensibles et ne doivent pas être utilisées pour prendre une décision individuelle de crédit. Même si leur inclusion améliorait une métrique, cela ne justifierait pas automatiquement leur emploi. Pour une institution de microfinance, le modèle devrait rester une aide à l'analyse, être audité par groupe, documenté et contestable par les personnes concernées. Les données étant simulées, les résultats ne décrivent pas des communautés réelles.

## Partie 6 — Profils de dépense (1 point)
Pour 2025, les dépenses sont agrégées par ménage sur l'ensemble de ses fêtes. On calcule les parts des cinq postes et l'effort de dépense, défini comme log(dépenses annuelles / revenu mensuel). Les variables sont standardisées avant K-means ; k est comparé par silhouette et interprétabilité.

In [ ]:
h25=hh[hh.annee.eq(2025)].copy()
expense_cols=["dep_alimentation_fcfa","dep_habillement_fcfa","dep_betail_fcfa",
              "dep_transport_ziara_fcfa","dep_cadeaux_dons_fcfa"]
annual=h25.groupby("menage_id").agg({**{c:"sum" for c in expense_cols},
    "depense_totale_fcfa":"sum","revenu_mensuel_fcfa":"first","taille_menage":"first",
    "region":"first","milieu":"first"}).reset_index()
annual["depenses_annuelles_fetes"]=annual[expense_cols].sum(axis=1)
for c in expense_cols:
    annual["part_"+c.replace("dep_","").replace("_fcfa","")]=annual[c]/annual["depenses_annuelles_fetes"].replace(0,np.nan)
annual["effort_log"]=np.log1p(annual["depenses_annuelles_fetes"]/annual["revenu_mensuel_fcfa"].replace(0,np.nan))
cluster_features=[c for c in annual.columns if c.startswith("part_")]+["effort_log","taille_menage"]
Z=SimpleImputer(strategy="median").fit_transform(annual[cluster_features])
Z=StandardScaler().fit_transform(Z)
sil=[]
for k in range(2,7):
    km=KMeans(n_clusters=k,random_state=RANDOM_STATE,n_init=10)
    labels=km.fit_predict(Z)
    sil.append({"k":k,"silhouette":silhouette_score(Z,labels,sample_size=min(3000,len(Z)),random_state=RANDOM_STATE)})
sil_df=pd.DataFrame(sil); display(sil_df)
best_k=int(sil_df.sort_values("silhouette",ascending=False).iloc[0]["k"])
km=KMeans(n_clusters=best_k,random_state=RANDOM_STATE,n_init=10)
annual["profil"]=km.fit_predict(Z)
profile=annual.groupby("profil").agg(menages=("menage_id","count"),
    depenses_mediane=("depenses_annuelles_fetes","median"),effort_median=("effort_log","median"),
    taille_mediane=("taille_menage","median"),part_alimentation=("part_alimentation","median"),
    part_habillement=("part_habillement","median"),part_betail=("part_betail","median"),
    part_transport_ziara=("part_transport_ziara","median"),part_cadeaux_dons=("part_cadeaux_dons","median"))
display(profile.round(3))

## Partie 7 — Plan 2026 et note de synthèse (1 point)
La grille demandée contient 365 × 10 × 15 = 54 750 lignes. On réutilise le modèle retenu après comparaison sur 2024, réentraîné sur 2016–2025. Pour la grille 2026, les variables connues à l'avance sont les dates, les métadonnées de magasin/catégorie et les promotions connues. Les promotions futures inconnues sont fixées à zéro : c'est une hypothèse, pas une information réelle.

In [ ]:
# Grille complète 2026 : 365 jours × 10 magasins × 15 catégories = 54 750 lignes.
dates26 = pd.date_range("2026-01-01", "2026-12-31", freq="D")
grid = pd.MultiIndex.from_product([dates26, shops.magasin_id, categories.categorie], names=["date", "magasin_id", "categorie"]).to_frame(index=False)
grid["annee"] = grid.date.dt.year
grid["mois"] = grid.date.dt.month
grid["jour_annee"] = grid.date.dt.dayofyear
grid["jour_semaine"] = grid.date.dt.dayofweek
grid["weekend"] = (grid.jour_semaine >= 5).astype(int)
grid["promo"] = 0  # hypothèse : promotions 2026 non encore connues
features26 = variables_calendrier(grid.date)
grid = pd.concat([grid.reset_index(drop=True), features26.drop(columns="date").reset_index(drop=True)], axis=1)
grid = grid.merge(shops[["magasin_id", "ville", "type_point_vente", "milieu"]], on="magasin_id", how="left")
grid = grid.merge(categories[["categorie", "famille", "unite"]], on="categorie", how="left")

# Choix du meilleur modèle sur validation 2024 ; entraînement final jusqu'à 2025.
train_all = model_df[model_df.annee.between(2016, 2025)].copy()
if wape(valid.quantite, pred_ridge) <= min(v[1] for v in hgb_candidates.values()):
    final_model = "Ridge log1p"
    model_2026 = clone(ridge).fit(train_all[features], np.log1p(train_all.quantite.clip(lower=0)))
    grid["quantite_prevue"] = np.expm1(model_2026.predict(grid[features])).clip(0)
else:
    final_model = "HGB " + best_loss
    # Codes de catégories construits sur les mêmes catégories pour apprentissage et prévision.
    full_for_codes = pd.concat([model_df[categorical_features], grid[categorical_features]], ignore_index=True)
    codes = {}
    for c in categorical_features:
        cats_c = pd.Index(full_for_codes[c].astype(str).unique())
        codes[c] = {v:i for i,v in enumerate(cats_c)}
    hf = train_all.copy()
    gh = grid.copy()
    for c in categorical_features:
        hf[c] = hf[c].astype(str).map(codes[c]).fillna(-1).astype(int)
        gh[c] = gh[c].astype(str).map(codes[c]).fillna(-1).astype(int)
    # Échantillon reproductible pour limiter le temps de calcul sur 10 ans de données.
    hf_sample = hf.sample(n=min(220000, len(hf)), random_state=RANDOM_STATE)
    y_sample = np.log1p(hf_sample.quantite.clip(lower=0)) if best_loss == "squared_error" else hf_sample.quantite.clip(lower=0)
    model_2026 = HistGradientBoostingRegressor(loss=best_loss, max_iter=45, max_leaf_nodes=20,
                    learning_rate=.08, l2_regularization=1.0, random_state=RANDOM_STATE)
    model_2026.fit(hf_sample[hgb_num_features], y_sample)
    pp = model_2026.predict(gh[hgb_num_features])
    grid["quantite_prevue"] = np.expm1(pp).clip(0) if best_loss == "squared_error" else pp.clip(0)
print("Modèle retenu pour le plan 2026 :", final_model, "| Grille :", grid.shape)

comparisons = []
for ev in ["Tabaski", "Magal de Touba", "Noël"]:
    e26 = calendar[(calendar.annee.eq(2026)) & calendar.evenement.eq(ev)]
    e25 = calendar[(calendar.annee.eq(2025)) & calendar.evenement.eq(ev)]
    if len(e26) and len(e25):
        d26, d25 = pd.Timestamp(e26.iloc[0].date_debut), pd.Timestamp(e25.iloc[0].date_debut)
        pred26 = grid[(grid.date >= d26-pd.Timedelta(days=7)) & (grid.date < d26)].quantite_prevue.sum()
        real25 = daily[(daily.date >= d25-pd.Timedelta(days=7)) & (daily.date < d25)].quantite.sum()
        comparisons.append({"événement":"Semaine avant "+ev, "réel 2025":real25, "prévision 2026":pred26,
                            "variation_vs_2025_pct":100*(pred26/real25-1) if real25 else np.nan})
comparisons = pd.DataFrame(comparisons)
display(comparisons.round(1))
grid.to_csv(BASE / "previsions_2026.csv", index=False)
print("Fichier créé : previsions_2026.csv")

### Note de synthèse à la direction (résumé automatique à compléter après lecture des résultats)
- **Objectif :** préparer les stocks 2026 à partir des ventes simulées 2016–2025 et du calendrier événementiel.
- **Recommandation opérationnelle :** utiliser les quantités prédites par magasin et catégorie comme point de départ, puis prévoir un stock de sécurité sur les catégories dont les pics historiques sont élevés. Pour les promotions, tester des scénarios séparés car leur fréquence augmente avant les fêtes.
- **Hypothèses :** calendrier 2026 tel que fourni ; promotions inconnues fixées à zéro ; modèle entraîné sur les années disponibles ; l'indice de prix est un indice pédagogique fondé sur le panier 2016.
- **Limites :** données simulées, ruptures de stock non observées, incertitude sur les dates lunaires locales, modèle sensible aux changements de prix et aux événements qui se chevauchent. Les prévisions ne sont pas une garantie de ventes.
- **Usage responsable :** ne pas déduire de comportements réels de groupes religieux ; ne pas utiliser les variables sensibles pour une décision de crédit.

## Bonus — biais des promotions
La variable `promo` est considérée connue à l'avance dans l'exercice, mais une promotion est plus fréquente avant les fêtes. Une simple association entre promotion et ventes confond donc l'effet de la réduction de prix avec la hausse saisonnière liée à la fête. Pour estimer un effet propre, il faudrait comparer des magasins/catégories/jours similaires, contrôler la distance à la fête et, idéalement, exploiter une affectation quasi-expérimentale ou randomisée.

## Déclaration d'utilisation d'un assistant IA
Un assistant IA a été utilisé pour aider à structurer le notebook et proposer du code. Les membres du binôme doivent vérifier les sorties, comprendre chaque cellule et être capables de justifier les choix à l'oral.